# Noble gas with the Lennard-Jones potential

This notebook shows how to simulate the simplest possible system for which a constant pressure and temperature (NpT) simulation makes sense.
The system consists of 199 argon atoms in a periodic box at $1\,\textrm{bar}$ and $293.15\,\textrm{K}$.

Because argon is a noble gas, the force field is relatively simple: no bonds need to be defined (unlike the water example in `01_water.ipynb`) and all interatomic interactions can be modeled with the [Lennard-Jones potential](https://en.wikipedia.org/wiki/Lennard-Jones_potential).

In [ ]:
%matplotlib widget
from sys import stdout

import matplotlib.pyplot as plt
import mdtraj
import nglview
import numpy as np
import openmm as mm
import pandas
from openmm import app, unit

## 1. Definition of the model

The Lennard-Jones parameters for argon, $\sigma = 3.419\,\textrm{Å}$ and $\epsilon/k_\textrm{B} = 117.8\,\textrm{K}$, were taken from J. A. White, *J. Chem. Phys.* **111**, 9352 (1999), [10.1063/1.479848](https://doi.org/10.1063/1.479848).
In that paper, they were chosen such that an approximate theory of the Lennard-Jones fluid reproduces the experimental critical temperature and density of argon.
A molecular dynamics simulation with the same parameters will not reproduce the critical point exactly, especially because the potential is truncated below.
The paper also lists several other parameter sets for argon, which shows that the Lennard-Jones potential is only an approximate model for a real noble gas.

In [ ]:
# Physical parameters
temperature = 293.15 * unit.kelvin
pressure = 1 * unit.bar
mass = 39.948 * unit.amu
sigma = 3.419 * unit.angstrom
epsilon = 117.8 * unit.kelvin * unit.BOLTZMANN_CONSTANT_kB * unit.AVOGADRO_CONSTANT_NA
print("epsilon:", epsilon.in_units_of(unit.kilojoule_per_mole))

# Simulation parameters
box_size = 150 * unit.angstrom  # initial value only
natom = 199
cutoff = 3 * sigma

## 2. The NpT molecular dynamics simulation

In the following cell, we define the `system`, `topology` and `force` objects.
Each of them keeps track of a specific aspect of the simulation we'd like to carry out:

- `system`: cell vectors, number of particles and their masses.

- `topology`: identity of the particles and related information (mostly irrelevant in this case): covalent bonds, residues and chains.
  All argon atoms are grouped into one residue in one chain.
  This extra information is needed to write out PDB files later.

- `force`: definition of the force field energy as function of particle positions and cell vectors.
  This example uses [NonbondedForce](https://docs.openmm.org/latest/api-python/generated/openmm.openmm.NonbondedForce.html) to define a Lennard-Jones potential, with a cutoff, a switching function and a long-range dispersion correction.

In [ ]:
# Define the OpenMM system with the argon atoms.
system = mm.System()
box_matrix = box_size * np.identity(3)
system.setDefaultPeriodicBoxVectors(*box_matrix)
for _iatom in range(natom):
    system.addParticle(mass)

# Define a relatively boring topology object.
# AR is the residue name for argon in the Protein Data Bank.
topology = app.Topology()
topology.setPeriodicBoxVectors(box_matrix)
chain = topology.addChain()
residue = topology.addResidue("AR", chain)
element_Ar = app.Element.getByAtomicNumber(18)
for _iatom in range(natom):
    topology.addAtom("Ar", element_Ar, residue)

# Define the force field as a "force" object to be added to the system.
force = mm.NonbondedForce()
force.setNonbondedMethod(mm.NonbondedForce.CutoffPeriodic)
for _iatom in range(natom):
    force.addParticle(0.0, sigma, epsilon)
force.setCutoffDistance(cutoff)
force.setUseSwitchingFunction(True)
force.setSwitchingDistance(0.8 * cutoff)
force.setUseDispersionCorrection(True)
force_index = system.addForce(force)

The following cell specifies more details of the NpT molecular dynamics simulation.
Conveying our choice of ensemble to OpenMM may not be very intuitive:

- The thermostat is set by selecting the [LangevinMiddleIntegrator](https://docs.openmm.org/latest/api-python/generated/openmm.openmm.LangevinMiddleIntegrator.html) to integrate the Langevin equations of motion.
  With this choice, a constant temperature is imposed by modeling the interaction of every atom with a heat bath, comprising two effects: (i) a friction force which leaks kinetic energy and (ii) random collisions that introduce kinetic energy.
  These two effects balance each other and reach an equilibrium when the atomistic degrees of freedom reach the desired temperature.

- The barostat is defined as a *force*, i.e. [MonteCarloBarostat](https://docs.openmm.org/latest/api-python/generated/openmm.openmm.MonteCarloBarostat.html) is added to the list of forces in the system.
  Here the same temperature as the thermostat must be given, in addition to the desired external pressure.
  With this option, the volume of the periodic box will be regularly rescaled to obtain an equilibrium between the internal pressure (generated by the argon atoms) and the external pressure.

The barostat is implemented as a force, mainly because the "forces" in an OpenMM `System` object can also perform other tasks than computing potential energies and atomic forces.
They may also modify properties of the system, such as cell vectors, atomic positions, atomic velocities, etc.

The last part of the code cell prepares the initial state of the system: random atomic positions, an energy minimization to remove close contacts between atoms, and random initial velocities.

In [ ]:
# Define the ensemble to be simulated in.
integrator = mm.LangevinMiddleIntegrator(temperature, 1 / unit.picosecond, 2 * unit.femtoseconds)
system.addForce(mm.MonteCarloBarostat(pressure, temperature))

# Define a simulation object.
simulation = app.Simulation(topology, system, integrator)

# Initialization steps before MD.
# - Assign random positions
rng = np.random.default_rng()
simulation.context.setPositions(
    rng.uniform(0, box_size / unit.angstrom, (natom, 3)) * unit.angstrom
)
# - Minimize the energy
simulation.minimizeEnergy()
# - Initialize velocities with random values at the target temperature.
simulation.context.setVelocitiesToTemperature(temperature)

The following cell defines all reporters and performs the MD simulation.

In [ ]:
# Remove existing reporters, in case this cell is executed more than once.
simulation.reporters = []

# Write the initial geometry as a PDB file.
positions = simulation.context.getState(getPositions=True).getPositions()
with open("ljinit.pdb", "w") as f:
    app.PDBFile.writeFile(simulation.topology, positions, f)

# Write a frame to the DCD trajectory every 100 steps.
simulation.reporters.append(app.DCDReporter("ljtraj.dcd", 100))

# Write scalar properties to a CSV file every 10 steps.
simulation.reporters.append(
    app.StateDataReporter(
        "ljscalars.csv",
        10,
        time=True,
        potentialEnergy=True,
        totalEnergy=True,
        temperature=True,
        volume=True,
    )
)

# Write scalar properties to screen every 1000 steps.
simulation.reporters.append(
    app.StateDataReporter(stdout, 1000, step=True, temperature=True, volume=True)
)

# Actually run the molecular dynamics simulation.
simulation.step(30000)

## 3. Analysis

A visualization of the trajectory is not very exciting: just argon atoms jumping around.
As you may guess, this is a simulation of argon gas.

Note that NGLView only shows the periodic box of the first frame.
The box size does change during the simulation, as can be seen in the volume plot below.

In [ ]:
traj = mdtraj.load("ljtraj.dcd", top="ljinit.pdb")
view = nglview.show_mdtraj(traj)
view.add_unitcell()
view

The temperature and volume as function of time reveal different time scales needed for the equilibration.

- The timescale of the thermostat is controlled by the friction coefficient, which is the second argument of [LangevinMiddleIntegrator](https://docs.openmm.org/latest/api-python/generated/openmm.openmm.LangevinMiddleIntegrator.html).
  Its inverse ($1\,\textrm{ps}$ in this example) is a measure for the time it takes the thermostat to correct deviations from the desired temperature.

- The rate of convergence of the barostat is controlled by the optional `frequency` argument, by default 25, of the [MonteCarloBarostat](https://docs.openmm.org/latest/api-python/generated/openmm.openmm.MonteCarloBarostat.html).
  This means that every 25 steps, an attempt is made to change the volume of the system.

As in the water example, the spurious characters in the header of the first column of the CSV file are removed after loading it.

In [ ]:
df = pandas.read_csv("ljscalars.csv")
df = df.rename(columns={'#"Time (ps)"': "Time (ps)"})
plt.close("temperature")
fig, ax = plt.subplots(num="temperature")
df.plot(kind="line", x="Time (ps)", y="Temperature (K)", ax=ax);

In [ ]:
plt.close("volume")
fig, ax = plt.subplots(num="volume")
df.plot(kind="line", x="Time (ps)", y="Box Volume (nm^3)", ax=ax);

**<span style="color:#A03;font-size:14pt">
&#x270B; HANDS-ON! &#x1F528;
</span>**

> - Slow down the thermostat and speed up the barostat. Can you make arbitrary changes without crashing the molecular dynamics run?
>
> - Change the pressure to $10\,\textrm{bar}$ and the temperature to $50\,\textrm{K}$.
>   Under these conditions, argon is a solid, because $50\,\textrm{K}$ is well below its [triple point](https://webbook.nist.gov/cgi/cbook.cgi?ID=C7440371&Mask=4) at $83.8\,\textrm{K}$ and $0.69\,\textrm{bar}$.
>   You will also have to change other parameters to make this work.
>   Do you see any signs in your simulation of a phase transition?